# Step 2 - Avaliação do modelo atual (champion)

**Objetivo 1 do case:** qual é a performance preditiva do modelo em produção, e quais métricas são mais adequadas para avaliá-lo?

O champion entrega um `score` de 0 a 100 (quanto maior, maior o risco). Os números deste notebook são a **referência** que o Challenger precisa superar.

## Imports e setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
LEGIT_COLOR, FRAUD_COLOR = "#2a78d6", "#eb6834"

## Regra de negócio: o Profit

Para cada transação, a decisão é **aprovar** ou **recusar**. Uma transação é recusada quando `score >= corte`.

| Decisão | Transação legítima | Fraude |
|---|---|---|
| Aprovada | ganha **10%** do `monto` (comissão) | perde **100%** do `monto` |
| Recusada | 0 | 0 |

Aprovar uma fraude custa o mesmo que a comissão de ~10 vendas legítimas do mesmo valor. Por isso o erro mais caro é deixar passar fraude, mas recusar demais também custa: cada legítima recusada é uma comissão perdida.

## Dados

In [ ]:
df = pd.read_csv('https://hr-projects-assets-prod.s3.amazonaws.com/3st1g8gm56o/ed953178414189325c980dc1a9214b97/data.csv')
df["fecha"] = pd.to_datetime(df["fecha"])

# Início do período de Test do split out-of-time (Step 3): usado no fim deste notebook
TEST_INICIO = "2020-04-13"

df[["score", "monto", "fecha", "fraude"]].describe()

## Por que a acurácia não serve

Com 5% de fraude, um "modelo" que aprova tudo já acerta 95% das transações e não pega nenhuma fraude.

In [ ]:
acuracia_aprovar_tudo = (df["fraude"] == 0).mean() * 100
print(f"Acurácia aprovando tudo: {acuracia_aprovar_tudo:.1f}% | fraudes detectadas: 0 de {df['fraude'].sum():,}")

## Métricas de separação (base completa)

- **ROC-AUC:** chance de uma fraude aleatória ter score maior que uma legítima aleatória (0,5 = aleatório).
- **PR-AUC:** precisão média ao longo dos cortes. A referência é a taxa de fraude (5%), não 0,5: foca na classe de fraude.
- **KS:** maior distância entre as distribuições acumuladas do score em fraudes e em legítimas. É o padrão de mercado em crédito e fraude.

In [ ]:
def metricas_separacao(dados):
    fraude = dados["fraude"]
    return pd.Series({
        "ROC-AUC": roc_auc_score(fraude, dados["score"]),
        "PR-AUC": average_precision_score(fraude, dados["score"]),
        "PR-AUC aleatório (taxa de fraude)": fraude.mean(),
        "KS": ks_2samp(dados.loc[fraude == 1, "score"], dados.loc[fraude == 0, "score"]).statistic,
    })

metricas_base = metricas_separacao(df)
metricas_base.round(3)

In [ ]:
fpr, tpr, _ = roc_curve(df["fraude"], df["score"])
precisao, recall, _ = precision_recall_curve(df["fraude"], df["score"])

# KS: % de fraudes e de legítimas com score >= corte, para cada corte
cortes = np.arange(0, 101)
pct_fraude = [(df.loc[df["fraude"] == 1, "score"] >= c).mean() * 100 for c in cortes]
pct_legit = [(df.loc[df["fraude"] == 0, "score"] >= c).mean() * 100 for c in cortes]
gap = np.array(pct_fraude) - np.array(pct_legit)
corte_ks = cortes[gap.argmax()]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 4.5))

ax1.plot(fpr, tpr, color=FRAUD_COLOR)
ax1.plot([0, 1], [0, 1], color="gray", linestyle="--")
ax1.set(title=f"ROC (AUC {metricas_base['ROC-AUC']:.3f})", xlabel="Taxa de falso positivo", ylabel="Recall (fraudes detectadas)")

ax2.plot(recall, precisao, color=FRAUD_COLOR)
ax2.axhline(df["fraude"].mean(), color="gray", linestyle="--")  # modelo aleatório
ax2.set(title=f"Precision-Recall (PR-AUC {metricas_base['PR-AUC']:.3f})", xlabel="Recall", ylabel="Precisão")

ax3.plot(cortes, pct_fraude, color=FRAUD_COLOR, label="% das fraudes com score >= corte")
ax3.plot(cortes, pct_legit, color=LEGIT_COLOR, label="% das legítimas com score >= corte")
ax3.vlines(corte_ks, pct_legit[corte_ks], pct_fraude[corte_ks], color="black", linestyle=":")
ax3.set(title=f"KS = {gap.max() / 100:.3f} (no corte {corte_ks})", xlabel="Corte do score", ylabel="% acumulado")
ax3.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Tabela de KS por decil

O score é dividido em 10 faixas com o mesmo número de transações, da de maior risco para a de menor. Para cada decil: taxa de fraude, % das fraudes capturadas até ali e o KS.

In [ ]:
def ks_por_decil(data, col, maior_e_pior=False):
    data = data[[col, "fraude"]].dropna()
    tabela = data.groupby(pd.qcut(data[col], 10, duplicates="drop"), observed=True)["fraude"].agg(
        fraudes="sum", total="size")
    if maior_e_pior:  # para o score: começa pelos decis de maior risco
        tabela = tabela.iloc[::-1]

    tabela["nao_fraudes"] = tabela["total"] - tabela["fraudes"]
    tabela["taxa_fraude_%"] = tabela["fraudes"] / tabela["total"] * 100
    tabela["fraude_acum_%"] = tabela["fraudes"].cumsum() / tabela["fraudes"].sum() * 100
    tabela["nao_fraude_acum_%"] = tabela["nao_fraudes"].cumsum() / tabela["nao_fraudes"].sum() * 100
    tabela["ks"] = tabela["fraude_acum_%"] - tabela["nao_fraude_acum_%"]
    return tabela.round(1)

decis_score = ks_por_decil(df, "score", maior_e_pior=True)
decis_score

In [ ]:
ax = decis_score["taxa_fraude_%"].plot(kind="bar", color=FRAUD_COLOR, figsize=(10, 4))
ax.axhline(df["fraude"].mean() * 100, color="gray", linestyle="--")  # média geral
ax.set(title="Taxa de fraude por decil do score (do maior risco para o menor)", xlabel="Faixa de score", ylabel="Taxa de fraude (%)")
plt.xticks(rotation=45)
plt.show()

**Leitura:**
- O decil de maior score (89–100) tem **21,8% de fraude** e sozinho captura **41,4% das fraudes**. Os três primeiros decis (score > 68) capturam ~70%: o score ordena bem o risco no topo.
- **O score não é monotônico:** os 3 decis de **menor** score (0–28) têm 2,8–4,7% de fraude, mais que os decis do meio (0,3–2,1%), e somam **~21% das fraudes**. Existe um grupo de fraudes que o champion classifica como muito seguras: um ponto cego que o Challenger pode tentar capturar. É esse grupo que deixa a curva ROC quase plana no meio.

## Profit por ponto de corte

A função abaixo aplica a regra de negócio: recusa quando `score >= corte` e soma o resultado das transações aprovadas. Antes de usar, ela é conferida num exemplo calculado à mão.

In [ ]:
def lucro(dados, corte, col_score="score"):
    aprovada = dados[col_score] < corte
    ganho = 0.10 * dados.loc[aprovada & (dados["fraude"] == 0), "monto"].sum()
    perda = dados.loc[aprovada & (dados["fraude"] == 1), "monto"].sum()
    return ganho - perda

# Conferência à mão: corte 50 aprova as linhas 1 e 3 → 0,10 × 100 − 30 = −20
exemplo = pd.DataFrame({"score": [20, 90, 40], "monto": [100.0, 50.0, 30.0], "fraude": [0, 1, 1]})
assert lucro(exemplo, 50) == -20
print("Exemplo:", lucro(exemplo, 50))

In [ ]:
cortes = np.arange(0, 102)  # corte 101 = aprovar tudo (nenhum score chega a 101)
curva = pd.DataFrame({
    "corte": cortes,
    "lucro": [lucro(df, c) for c in cortes],
    "taxa_recusa_%": [(df["score"] >= c).mean() * 100 for c in cortes],
}).set_index("corte")

lucro_aprovar_tudo = lucro(df, 101)
melhor_corte = curva["lucro"].idxmax()

ax = curva["lucro"].plot(color=FRAUD_COLOR, figsize=(10, 4))
ax.axhline(lucro_aprovar_tudo, color="gray", linestyle="--", label="aprovar tudo")
ax.axvline(melhor_corte, color="black", linestyle=":", label=f"melhor corte = {melhor_corte}")
ax.set(title="Profit por ponto de corte do champion (recusa quando score >= corte)", xlabel="Corte do score", ylabel="Profit")
ax.legend()
plt.show()

In [ ]:
def resumo_corte(dados, corte, col_score="score"):
    recusada = dados[col_score] >= corte
    fraude = dados["fraude"] == 1
    return pd.Series({
        "corte": corte,
        "lucro": lucro(dados, corte, col_score),
        "lucro_aprovar_tudo": lucro(dados, np.inf, col_score),
        "taxa_recusa_%": recusada.mean() * 100,
        "recall_%": (recusada & fraude).sum() / fraude.sum() * 100,      # fraudes recusadas / total de fraudes
        "precisao_%": (recusada & fraude).sum() / recusada.sum() * 100,  # fraudes entre as recusadas
    })

resumo_base = resumo_corte(df, melhor_corte)
resumo_base.round(2)

## Mesmas métricas no período de Test

O Challenger será avaliado no período de Test do split out-of-time (13/04 a 21/04). Para a comparação do notebook `challenger.ipynb` ser justa, o champion é medido no mesmo período, com o mesmo corte encontrado acima.

In [ ]:
test = df[df["fecha"] >= TEST_INICIO]

referencia_champion = pd.DataFrame({
    "Base completa": pd.concat([metricas_separacao(df), resumo_corte(df, melhor_corte)]),
    "Test (13/04–21/04)": pd.concat([metricas_separacao(test), resumo_corte(test, melhor_corte)]),
})
referencia_champion.round(3)

## Conclusão: performance do champion (objetivo 1)

| Métrica | Base completa | Test (13/04–21/04) |
|---|---|---|
| ROC-AUC | 0,726 | 0,733 |
| PR-AUC (aleatório = taxa de fraude) | 0,176 (0,050) | 0,229 (0,044) |
| KS | 0,432 | 0,463 |
| Melhor corte (score >= 76 → recusa) | Profit 264.929 | Profit 69.826 |
| Profit aprovando tudo | 50.849 | 19.626 |
| Taxa de recusa no corte | 22,4% | 17,0% |
| Recall / precisão no corte | 63,0% / 14,0% | 61,1% / 15,9% |

**Leitura**
- O champion **separa bem o topo do risco**: o decil de maior score captura 41% das fraudes, e o KS de 0,43 é bem maior que o de qualquer variável numérica sozinha (`f`, 0,28). Mas a `o` sozinha já chega a 0,45 (EDA).
- Ele tem um **ponto cego**: ~21% das fraudes estão nos 3 decis de menor score, como se fossem transações muito seguras.
- No melhor corte (recusar `score >= 76`), o Profit é **~5× o de aprovar tudo** (264,9 mil vs 50,8 mil), recusando 22% das transações e barrando 63% das fraudes.
- No período de Test as métricas de separação são até um pouco melhores, mas a **taxa de recusa cai para 17%** com o mesmo corte: a distribuição do score desceu em abril (média de ~51 para ~45, ver EDA). O corte que vale para a decisão é escolhido na validação (`challenger.ipynb`), não na base completa.

**Por que essas métricas**
- **Acurácia não serve:** aprovar tudo já dá 95% de acurácia sem pegar nenhuma fraude.
- **PR-AUC em vez de só ROC-AUC:** com 5% de fraude, a ROC-AUC é inflada pelas muitas legítimas fáceis de separar. A PR-AUC mede o desempenho na classe de fraude, e a referência dela é 0,05, não 0,5.
- **KS:** é o padrão de mercado em crédito e fraude e é fácil de ler ("no melhor corte, separa 43 p.p. mais fraude que legítima").
- **Profit decide:** as métricas acima medem separação; a decisão de negócio (onde cortar) é tomada pelo Profit, que pesa o valor de cada transação.

**Referência para o Challenger (`challenger.ipynb`):** a coluna Test da tabela acima.